# Matrix Multiplication on CPU: Optimizations and Insights

In this notebook, we explore CPU implementations of matrix multiplication.
We analyze various optimization techniques including:

- **Naive implementation**
- **Tiled execution**
- **Unrolled loop**

Each method aims to reduce global memory access and increase computation throughput.

## 📋 Lab 7 CPU — To-Do Summary (4 items)

Replace every `None` with the correct expression.
Use the `# Think about:` comments as your guide — not direct answers.

| # | Location | What to implement |
|---|----------|-------------------|
| 1 | `benchmark()` | Record the start time before calling the function |
| 2 | `benchmark()` | Compute average time from the collected list |
| 3 | `matmul_naive_cpu()` | Compute the dot product accumulation for one output element |
| 4 | Benchmarking cell | Add NumPy's built-in matmul as a baseline comparison |


## Preparing the required packages and initialisations


In [ ]:
import numpy as np
import torch
import time
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
import time

def benchmark(func, A, B, label, runs=3):
    """Time a matmul function averaged over multiple runs."""
    times = []
    for _ in range(runs):
        # To Do 1: record the wall-clock time at the start of this run
        # Think about: which time module function gives you the current time in seconds?
        start = None
        _ = func(A, B)
        times.append(time.time() - start)

    # To Do 2: compute the average time from the collected list
    # Think about: what arithmetic operation gives you the mean of a list?
    avg_time = None

    print(f"{label}: {avg_time:.4f} seconds")
    return avg_time


## Naive implementation

In [ ]:
import numpy as np

def matmul_naive_cpu(A, B):
    """Naive O(N^3) matrix multiplication using three nested loops."""
    C = np.zeros((A.shape[0], B.shape[1]))
    for i in range(A.shape[0]):
        for j in range(B.shape[1]):
            for k in range(A.shape[1]):
                # To Do 3: accumulate the contribution of A[i,k] * B[k,j] into C[i,j]
                # Think about: C[i,j] is a running sum — what operation extends a running sum?
                C[i, j] = None
    return C


## Tiled execution

In [ ]:
def matmul_tiled_cpu(A, B, tile_size=32):
    """Performs matrix multiplication with tiling."""
    C = np.zeros((A.shape[0], B.shape[1]))
    for i in range(0, A.shape[0], tile_size):
        for j in range(0, B.shape[1], tile_size):
            for k in range(0, A.shape[1], tile_size):
                for ii in range(i, min(i + tile_size, A.shape[0])):
                    for jj in range(j, min(j + tile_size, B.shape[1])):
                        for kk in range(k, min(k + tile_size, A.shape[1])):
                            C[ii, jj] += A[ii, kk] * B[kk, jj]
    return C

## Unrolled loop

In [ ]:
def matmul_unrolled_cpu(A, B):
  """Performs matrix multiplication with loop unrolling."""
  C = np.zeros((A.shape[0], B.shape[1]))
  for i in range(A.shape[0]):
    for j in range(B.shape[1]):
      sum_val = 0
      for k in range(0, A.shape[1], 4): # Unroll by 4
        sum_val += A[i, k] * B[k, j]
        sum_val += A[i,k+1] * B[k+1, j]
        sum_val += A[i,k+2] * B[k+2, j]
        sum_val += A[i,k+3] * B[k+3, j]
      C[i, j] = sum_val
  return C

## Benchmarking

In [ ]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

N = 32
assert N % 4 == 0
A = np.random.randn(N, N).astype(np.float32)
B = np.random.randn(N, N).astype(np.float32)

results = {}
results['CPU (naive)']    = benchmark(matmul_naive_cpu,    A, B, 'CPU (naive)')
results['CPU (tiled)']    = benchmark(matmul_tiled_cpu,    A, B, 'CPU (tiled)')
results['CPU (unrolled)'] = benchmark(matmul_unrolled_cpu, A, B, 'CPU (unrolled)')

# To Do 4: add NumPy's built-in matmul as a baseline so students can see
# how much faster an optimised BLAS routine is vs the hand-coded loops.
# Think about: numpy has a function that multiplies two matrices — what is it called?
#              Store its result in results['NumPy (baseline)'].
results['NumPy (baseline)'] = None

df = pd.DataFrame.from_dict(results, orient='index', columns=['Execution Time (s)'])
df.sort_values('Execution Time (s)', inplace=True)
df.plot(kind='barh', legend=False, figsize=(8, 4),
        title='CPU Matrix Multiplication Performance')
plt.xlabel('Time (seconds)')
plt.grid(True)
plt.tight_layout()
plt.show()
